# Recipe Dataset Loading and Preprocessing

**Purpose**: Download and preprocess real recipe datasets for training/fine-tuning GPT-2 model

**Task**: T019 [P1] [US1] - Load Recipe Dataset

**Dataset Sources**: 
- **RecipeNLG** (2.2M+ recipes) - Primary choice
- **Food.com** (500k+ recipes) - Alternative
- **Recipe1M+** (1M+ recipes with images) - Advanced option

**Outputs**:
- Downloaded raw dataset to `data/raw/recipes/`
- Cleaned recipe dataset saved to `data/processed/recipes/`
- Training and validation splits
- Dataset statistics and analysis

## 1. Environment Setup

In [8]:
import os
import json
import numpy as np
import pandas as pd
from pathlib import Path
import re
from typing import List, Dict, Optional
import warnings
warnings.filterwarnings('ignore')

# For downloading datasets
import urllib.request
import zipfile
import gzip
import shutil

# Set random seed for reproducibility
np.random.seed(42)

print("✅ Packages imported successfully")

✅ Packages imported successfully


## 2. Configure Paths

In [9]:
# Project directories
PROJECT_ROOT = Path.cwd().parent.parent
DATA_RAW = PROJECT_ROOT / "data" / "raw" / "recipes"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed" / "recipes"

# Create directories
DATA_RAW.mkdir(parents=True, exist_ok=True)
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

print(f"📁 Raw data directory: {DATA_RAW}")
print(f"📁 Processed data directory: {DATA_PROCESSED}")

📁 Raw data directory: c:\Users\Champion\Documents\GitHub\cAIuldron\data\raw\recipes
📁 Processed data directory: c:\Users\Champion\Documents\GitHub\cAIuldron\data\processed\recipes


## 3. Load Your Dataset

**Expected file location**: `data/raw/recipes/`

**Supported formats**:
- ZIP archives (will auto-extract)
- CSV files
- Excel files (.xlsx, .xls)

**Your dataset**: RecipeNLG
- File: `dataset.zip` (~600 MB)
- Contains: `dataset/full_dataset.csv` (2.3 GB, 2.23M recipes)
- Format: CSV with JSON arrays

The notebook will automatically:
1. Detect and extract ZIP files
2. Find CSV/Excel files
3. Load and process data

## 4. Alternative: Use Kaggle Food.com Dataset

If RecipeNLG download fails, you can use Food.com dataset from Kaggle:
1. Download from: https://www.kaggle.com/datasets/shuyangli94/food-com-recipes-and-user-interactions
2. Save `RAW_recipes.csv` to `data/raw/recipes/`
3. Run the cell below instead

In [10]:
# Step 1: Check for ZIP files and extract them
print("🔍 Checking for ZIP files...\n")

zip_files = list(DATA_RAW.glob("*.zip"))

if zip_files:
    print(f"✅ Found {len(zip_files)} ZIP file(s):")
    for zf in zip_files:
        print(f"   - {zf.name} ({zf.stat().st_size / (1024**2):.1f} MB)")
    
    # Extract each ZIP file
    for zip_file in zip_files:
        print(f"\n📦 Extracting: {zip_file.name}...")
        
        try:
            import zipfile
            
            with zipfile.ZipFile(zip_file, 'r') as zip_ref:
                # List contents
                file_list = zip_ref.namelist()
                print(f"   Contents: {len(file_list)} file(s)")
                
                # Show files
                print(f"   Files inside:")
                for i, fname in enumerate(file_list[:10], 1):
                    file_info = zip_ref.getinfo(fname)
                    size_mb = file_info.file_size / (1024**2)
                    print(f"      {i}. {fname} ({size_mb:.1f} MB)")
                if len(file_list) > 10:
                    print(f"      ... and {len(file_list) - 10} more files")
                
                # Extract all
                print(f"\n   Extracting to: {DATA_RAW}")
                zip_ref.extractall(DATA_RAW)
                
            print(f"   ✅ Extraction complete!\n")
            
        except Exception as e:
            print(f"   ❌ Extraction failed: {e}\n")
else:
    print("   No ZIP files found\n")

# Step 2: Check for all possible dataset files (including in subdirectories)
print("🔍 Searching for recipe datasets...\n")

# Check in main directory and subdirectories
alternative_files = [
    # Main directory
    DATA_RAW / "RAW_recipes.csv",
    DATA_RAW / "recipenlg_full.csv",
    DATA_RAW / "recipes.csv",
    DATA_RAW / "full_dataset.csv",
    # In dataset/ subdirectory (from ZIP extraction)
    DATA_RAW / "dataset" / "full_dataset.csv",
    DATA_RAW / "dataset" / "recipes.csv",
    # Excel formats
    DATA_RAW / "RAW_recipes.xlsx",
    DATA_RAW / "recipes.xlsx",
]

# Find which dataset is available
dataset_file = None
dataset_type = None
is_excel = False

for file in alternative_files:
    if file.exists():
        dataset_file = file
        is_excel = file.suffix in ['.xlsx', '.xls']
        
        # Determine dataset type
        if "RAW_recipes" in file.name:
            dataset_type = "food.com"
        elif "full_dataset" in file.name or "dataset" in str(file.parent):
            dataset_type = "recipenlg"
        else:
            dataset_type = "generic"
        
        print(f"✅ Found dataset: {file.name}")
        if file.parent.name != "recipes":
            print(f"   Location: {file.parent.name}/{file.name}")
        print(f"   Type: {dataset_type}")
        print(f"   Format: {'Excel' if is_excel else 'CSV'}")
        print(f"   Size: {file.stat().st_size / (1024**2):.1f} MB")
        
        # For very large files, estimate row count
        if file.stat().st_size > 1000 * 1024 * 1024:  # > 1GB
            estimated_rows = file.stat().st_size / 1024  # rough estimate
            print(f"   Estimated recipes: ~{estimated_rows:,.0f}")
        
        break

# Step 3: Final check
if not dataset_file:
    print("\n" + "="*80)
    print("⚠️ NO DATASET FOUND")
    print("="*80)
    print("\n💡 Please place your dataset file in the data directory:\n")
    print(f"📁 Directory: {DATA_RAW}\n")
    print("✅ Supported formats:")
    print("   - ZIP archives: *.zip (will auto-extract)")
    print("   - CSV files: *.csv")
    print("   - Excel files: *.xlsx, *.xls\n")
    print("📋 Expected filenames:")
    print("   - dataset.zip (RecipeNLG)")
    print("   - archive.zip (Food.com from Kaggle)")
    print("   - RAW_recipes.csv (Food.com)")
    print("   - full_dataset.csv (RecipeNLG)")
    print("\n" + "="*80)
else:
    print(f"\n✅ Ready to process: {dataset_file.name}")
    print(f"   Type: {dataset_type}")
    print(f"   Format: {'Excel' if is_excel else 'CSV'}")
    print(f"\n💡 Note: Large dataset detected!")
    print(f"   The notebook will load a sample (10,000 rows) for testing.")
    print(f"   To process the full dataset, modify the 'nrows' parameter later.")

🔍 Checking for ZIP files...

✅ Found 1 ZIP file(s):
   - dataset.zip (591.9 MB)

📦 Extracting: dataset.zip...
   Contents: 2 file(s)
   Files inside:
      1. dataset/ (0.0 MB)
      2. dataset/full_dataset.csv (2188.7 MB)

   Extracting to: c:\Users\Champion\Documents\GitHub\cAIuldron\data\raw\recipes
   ✅ Extraction complete!

🔍 Searching for recipe datasets...

✅ Found dataset: full_dataset.csv
   Location: dataset/full_dataset.csv
   Type: recipenlg
   Format: CSV
   Size: 2188.7 MB
   Estimated recipes: ~2,241,192

✅ Ready to process: full_dataset.csv
   Type: recipenlg
   Format: CSV

💡 Note: Large dataset detected!
   The notebook will load a sample (10,000 rows) for testing.
   To process the full dataset, modify the 'nrows' parameter later.


In [11]:
if dataset_file and dataset_file.exists():
    print(f"📖 Loading dataset: {dataset_file.name}\n")
    
    # Determine file type
    file_extension = dataset_file.suffix.lower()
    
    try:
        # Load dataset based on file type
        if file_extension == '.csv':
            print("   Format: CSV")
            print("   Loading (sample first 10,000 rows for testing)...")
            print("   💡 To load full dataset, remove 'nrows=10000' parameter\n")
            
            # Load CSV (sample first for speed)
            # For full dataset, remove nrows parameter
            df_raw = pd.read_csv(dataset_file, nrows=10000)
            
        elif file_extension in ['.xlsx', '.xls']:
            print(f"   Format: Excel ({file_extension})")
            print("   ⚠️ Excel files are slower to read than CSV")
            
            # Check if openpyxl is installed
            try:
                import openpyxl
            except ImportError:
                print("   Installing openpyxl...")
                import subprocess
                subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "openpyxl"])
            
            print("   Loading (sample first 10,000 rows for testing)...")
            print("   💡 To load full dataset, remove 'nrows=10000' parameter\n")
            
            # Load Excel (sample first for speed)
            df_raw = pd.read_excel(dataset_file, nrows=10000, engine='openpyxl')
            
        else:
            raise ValueError(f"Unsupported file format: {file_extension}")
        
        print(f"✅ Loaded {len(df_raw):,} recipes")
        print(f"\n📊 Dataset shape: {df_raw.shape}")
        print(f"   Rows: {df_raw.shape[0]:,}")
        print(f"   Columns: {df_raw.shape[1]}")
        
        print(f"\n📋 Columns found:")
        for i, col in enumerate(df_raw.columns, 1):
            print(f"   {i}. {col}")
        
        print(f"\n🔍 First few rows:")
        print(df_raw.head(3))
        
        # Show sample recipe
        print(f"\n" + "="*80)
        print("📝 Sample Recipe:")
        print("="*80)
        sample = df_raw.iloc[0]
        for col in df_raw.columns:
            value = str(sample[col])
            if len(value) > 200:
                value = value[:200] + "..."
            print(f"\n{col}:")
            print(f"  {value}")
        
        print("\n" + "="*80)
        
    except Exception as e:
        print(f"❌ Error loading dataset: {e}")
        print("\nPlease check:")
        print("1. File is not corrupted")
        print("2. File has proper format (CSV or Excel)")
        print("3. File contains recipe data")
        df_raw = None
        
else:
    print("❌ No dataset available. Please download a dataset first.")
    df_raw = None

📖 Loading dataset: full_dataset.csv

   Format: CSV
   Loading (sample first 10,000 rows for testing)...
   💡 To load full dataset, remove 'nrows=10000' parameter

✅ Loaded 10,000 recipes

📊 Dataset shape: (10000, 7)
   Rows: 10,000
   Columns: 7

📋 Columns found:
   1. Unnamed: 0
   2. title
   3. ingredients
   4. directions
   5. link
   6. source
   7. NER

🔍 First few rows:
   Unnamed: 0                  title  \
0           0    No-Bake Nut Cookies   
1           1  Jewell Ball'S Chicken   
2           2            Creamy Corn   

                                         ingredients  \
0  ["1 c. firmly packed brown sugar", "1/2 c. eva...   
1  ["1 small jar chipped beef, cut up", "4 boned ...   
2  ["2 (16 oz.) pkg. frozen corn", "1 (8 oz.) pkg...   

                                          directions  \
0  ["In a heavy 2-quart saucepan, mix brown sugar...   
1  ["Place chipped beef on bottom of baking dish....   
2  ["In a slow cooker, combine all ingredients. C...   

       

## 6. Clean and Standardize Data

In [12]:
def clean_recipe_data(df: pd.DataFrame, dataset_type: str) -> pd.DataFrame:
    """
    Clean and standardize recipe data
    
    Args:
        df (DataFrame): Raw recipe dataframe
        dataset_type (str): Type of dataset (recipenlg, food.com, etc.)
    
    Returns:
        DataFrame: Cleaned recipe data
    """
    print(f"🧹 Cleaning {dataset_type} dataset...\n")
    
    df_clean = df.copy()
    
    # Map columns based on dataset type
    if dataset_type == "food.com":
        column_mapping = {
            'name': 'recipe_title',
            'minutes': 'cooking_time_minutes',
            'n_steps': 'num_steps',
            'steps': 'instructions',
            'ingredients': 'ingredients',
            'n_ingredients': 'num_ingredients'
        }
    elif dataset_type == "recipenlg":
        column_mapping = {
            'title': 'recipe_title',
            'directions': 'instructions',
            'ingredients': 'ingredients',
            'NER': 'ingredient_tags'
        }
    else:
        # Generic mapping - try to auto-detect
        column_mapping = {}
        if 'title' in df_clean.columns:
            column_mapping['title'] = 'recipe_title'
        elif 'name' in df_clean.columns:
            column_mapping['name'] = 'recipe_title'
        
        if 'directions' in df_clean.columns:
            column_mapping['directions'] = 'instructions'
        elif 'steps' in df_clean.columns:
            column_mapping['steps'] = 'instructions'
    
    # Rename columns
    df_clean = df_clean.rename(columns=column_mapping)
    
    # Remove rows with missing critical data
    initial_count = len(df_clean)
    required_cols = ['recipe_title', 'ingredients', 'instructions']
    existing_required = [col for col in required_cols if col in df_clean.columns]
    
    if existing_required:
        df_clean = df_clean.dropna(subset=existing_required)
        print(f"   Removed {initial_count - len(df_clean):,} rows with missing data")
    
    # Parse ingredients and instructions if stored as strings
    def parse_json_field(field):
        if pd.isna(field):
            return []
        if isinstance(field, list):
            return field
        if isinstance(field, str):
            try:
                # Try parsing as JSON
                import ast
                parsed = ast.literal_eval(field)
                if isinstance(parsed, list):
                    return parsed
                return [str(parsed)]
            except:
                # Split by common delimiters
                return [x.strip() for x in re.split(r'[\n;]', field) if x.strip()]
        return []
    
    if 'ingredients' in df_clean.columns:
        print("   Parsing ingredients...")
        df_clean['ingredients'] = df_clean['ingredients'].apply(parse_json_field)
    
    if 'instructions' in df_clean.columns:
        print("   Parsing instructions...")
        df_clean['instructions'] = df_clean['instructions'].apply(parse_json_field)
    
    # Filter out recipes with too few ingredients or steps
    before_filter = len(df_clean)
    df_clean = df_clean[
        (df_clean['ingredients'].apply(len) >= 3) &
        (df_clean['instructions'].apply(len) >= 3)
    ]
    filtered_out = before_filter - len(df_clean)
    if filtered_out > 0:
        print(f"   Removed {filtered_out:,} recipes with <3 ingredients or <3 steps")
    
    print(f"\n✅ Cleaning complete: {len(df_clean):,} recipes")
    return df_clean

if 'df_raw' in locals() and df_raw is not None and dataset_type:
    df_clean = clean_recipe_data(df_raw, dataset_type)
    print(f"\n📊 Cleaned dataset shape: {df_clean.shape}")
    print(f"   Columns: {list(df_clean.columns)}")
else:
    print("⚠️ No raw data to clean")

🧹 Cleaning recipenlg dataset...

   Removed 0 rows with missing data
   Parsing ingredients...
   Parsing instructions...
   Removed 2,087 recipes with <3 ingredients or <3 steps

✅ Cleaning complete: 7,913 recipes

📊 Cleaned dataset shape: (7913, 7)
   Columns: ['Unnamed: 0', 'recipe_title', 'ingredients', 'instructions', 'link', 'source', 'ingredient_tags']


## 7. Extract Main Ingredient from Each Recipe

In [13]:
def extract_main_ingredient(ingredients: List[str]) -> str:
    """
    Extract likely main ingredient from ingredients list
    
    Args:
        ingredients (list): List of ingredient strings
    
    Returns:
        str: Main ingredient name
    """
    # Common main ingredients (proteins, vegetables)
    main_proteins = [
        'chicken', 'beef', 'pork', 'salmon', 'tuna', 'shrimp', 'fish',
        'turkey', 'lamb', 'duck', 'tofu', 'eggs'
    ]
    
    main_vegetables = [
        'tomato', 'potato', 'carrot', 'broccoli', 'spinach', 'mushroom',
        'onion', 'garlic', 'pepper', 'zucchini', 'eggplant'
    ]
    
    # Search for main proteins first (higher priority)
    for ing_text in ingredients:
        ing_lower = ing_text.lower()
        for protein in main_proteins:
            if protein in ing_lower:
                # Extract specific type (e.g., "chicken breast" vs just "chicken")
                if 'breast' in ing_lower and 'chicken' in ing_lower:
                    return 'chicken breast'
                elif 'thigh' in ing_lower and 'chicken' in ing_lower:
                    return 'chicken thigh'
                elif 'ground' in ing_lower and 'beef' in ing_lower:
                    return 'ground beef'
                else:
                    return protein
    
    # Search for main vegetables
    for ing_text in ingredients:
        ing_lower = ing_text.lower()
        for veg in main_vegetables:
            if veg in ing_lower:
                return veg
    
    # Default: use first ingredient
    if ingredients:
        # Extract just the ingredient name (remove quantities)
        first = ingredients[0].lower()
        # Remove numbers and common units
        clean = re.sub(r'\d+', '', first)
        clean = re.sub(r'(cup|tbsp|tsp|oz|lb|kg|g|ml|l)s?', '', clean)
        clean = clean.strip()
        return clean.split()[0] if clean.split() else 'unknown'
    
    return 'unknown'

if 'df_clean' in locals():
    # Extract main ingredient for each recipe
    print("🔍 Extracting main ingredients...\n")
    df_clean['ingredient'] = df_clean['ingredients'].apply(extract_main_ingredient)
    
    print(f"✅ Main ingredients extracted")
    print(f"\n📊 Top 20 ingredients:")
    print(df_clean['ingredient'].value_counts().head(20))
else:
    print("⚠️ No cleaned data available")

🔍 Extracting main ingredients...

✅ Main ingredients extracted

📊 Top 20 ingredients:
ingredient
eggs              1768
c.                 713
chicken            597
onion              542
/                  514
ground beef        379
pepper             309
chicken breast     278
(                  268
.                  222
beef               207
potato             185
tomato             162
pork               160
p.                 125
garlic             124
broccoli            92
are                 89
carrot              83
shrimp              81
Name: count, dtype: int64


## 8. Add Metadata (Cuisine, Difficulty)

In [14]:
def infer_cuisine(title: str, ingredients: List[str]) -> str:
    """
    Infer cuisine type from recipe title and ingredients
    
    Args:
        title (str): Recipe title
        ingredients (list): Ingredients list
    
    Returns:
        str: Cuisine type
    """
    text = (title + " " + " ".join(ingredients)).lower()
    
    # Cuisine keywords
    cuisines = {
        'italian': ['italian', 'pasta', 'spaghetti', 'pizza', 'parmesan', 'basil', 'marinara'],
        'asian': ['asian', 'soy sauce', 'sesame', 'ginger', 'stir fry', 'wok'],
        'chinese': ['chinese', 'szechuan', 'mandarin', 'canton'],
        'japanese': ['japanese', 'sushi', 'teriyaki', 'miso', 'sake', 'mirin'],
        'thai': ['thai', 'curry paste', 'fish sauce', 'lemongrass', 'galangal'],
        'indian': ['indian', 'curry', 'garam masala', 'turmeric', 'tikka', 'tandoori', 'naan'],
        'mexican': ['mexican', 'taco', 'burrito', 'salsa', 'cilantro', 'cumin', 'tortilla'],
        'french': ['french', 'provence', 'bourguignon', 'confit', 'ratatouille'],
        'mediterranean': ['mediterranean', 'olive oil', 'feta', 'hummus', 'tahini'],
        'korean': ['korean', 'kimchi', 'gochujang', 'bulgogi'],
    }
    
    for cuisine, keywords in cuisines.items():
        if any(keyword in text for keyword in keywords):
            return cuisine
    
    return 'american'  # Default

def infer_difficulty(num_ingredients: int, num_steps: int, cooking_time: Optional[int] = None) -> str:
    """
    Infer difficulty level
    
    Args:
        num_ingredients (int): Number of ingredients
        num_steps (int): Number of cooking steps
        cooking_time (int, optional): Cooking time in minutes
    
    Returns:
        str: Difficulty level (easy/medium/hard)
    """
    score = 0
    
    # More ingredients = harder
    if num_ingredients > 12:
        score += 2
    elif num_ingredients > 8:
        score += 1
    
    # More steps = harder
    if num_steps > 10:
        score += 2
    elif num_steps > 6:
        score += 1
    
    # Longer time = harder
    if cooking_time:
        if cooking_time > 90:
            score += 2
        elif cooking_time > 45:
            score += 1
    
    if score >= 4:
        return 'hard'
    elif score >= 2:
        return 'medium'
    else:
        return 'easy'

if 'df_clean' in locals():
    print("🏷️ Adding metadata...\n")
    
    # Infer cuisine
    df_clean['cuisine'] = df_clean.apply(
        lambda row: infer_cuisine(row['recipe_title'], row['ingredients']),
        axis=1
    )
    
    # Infer difficulty
    df_clean['difficulty'] = df_clean.apply(
        lambda row: infer_difficulty(
            len(row['ingredients']),
            len(row['instructions']),
            row.get('cooking_time_minutes')
        ),
        axis=1
    )
    
    # Add servings if not present
    if 'servings' not in df_clean.columns:
        df_clean['servings'] = 4  # Default
    
    # Add cooking time if not present
    if 'cooking_time_minutes' not in df_clean.columns:
        df_clean['cooking_time_minutes'] = 30  # Default
    
    print("✅ Metadata added")
    print(f"\n📊 Cuisine distribution:")
    print(df_clean['cuisine'].value_counts())
    print(f"\n📊 Difficulty distribution:")
    print(df_clean['difficulty'].value_counts())
else:
    print("⚠️ No data available")

🏷️ Adding metadata...

✅ Metadata added

📊 Cuisine distribution:
cuisine
american         6327
italian           644
asian             328
mexican           264
french            119
indian             90
chinese            72
mediterranean      61
japanese            7
thai                1
Name: count, dtype: int64

📊 Difficulty distribution:
difficulty
easy      6354
medium    1507
hard        52
Name: count, dtype: int64


## 9. Format for GPT-2 Training

In [15]:
def format_recipe_for_training(recipe: Dict) -> str:
    """
    Format recipe as structured text for GPT-2 training
    
    Args:
        recipe (dict): Recipe dictionary
    
    Returns:
        str: Formatted recipe text
    """
    # Join ingredients list
    if isinstance(recipe['ingredients'], list):
        ingredients_str = "; ".join(recipe['ingredients'])
    else:
        ingredients_str = str(recipe['ingredients'])
    
    # Join instructions
    if isinstance(recipe['instructions'], list):
        instructions_str = " ".join([
            f"{i+1}. {step}" for i, step in enumerate(recipe['instructions'])
        ])
    else:
        instructions_str = str(recipe['instructions'])
    
    # Create structured format
    formatted = f"""<INGREDIENT> {recipe['ingredient']}
<TITLE> {recipe['recipe_title']}
<CUISINE> {recipe['cuisine']}
<DIFFICULTY> {recipe['difficulty']}
<TIME> {recipe['cooking_time_minutes']} minutes
<SERVINGS> {recipe['servings']}
<INGREDIENTS> {ingredients_str}
<INSTRUCTIONS> {instructions_str}
<END>"""
    
    return formatted

if 'df_clean' in locals():
    print("📝 Formatting recipes for training...\n")
    
    # Convert to dict format
    recipes_list = df_clean.to_dict('records')
    
    # Format all recipes
    formatted_recipes = [format_recipe_for_training(recipe) for recipe in recipes_list]
    df_clean['formatted_text'] = formatted_recipes
    
    print(f"✅ Formatted {len(formatted_recipes):,} recipes")
    print("\n📝 Example formatted recipe:")
    print("=" * 80)
    print(formatted_recipes[0][:500] + "...")  # Show first 500 chars
    print("=" * 80)
else:
    print("⚠️ No data to format")

📝 Formatting recipes for training...

✅ Formatted 7,913 recipes

📝 Example formatted recipe:
<INGREDIENT> c.
<TITLE> No-Bake Nut Cookies
<CUISINE> american
<DIFFICULTY> easy
<TIME> 30 minutes
<SERVINGS> 4
<INGREDIENTS> 1 c. firmly packed brown sugar; 1/2 c. evaporated milk; 1/2 tsp. vanilla; 1/2 c. broken nuts (pecans); 2 Tbsp. butter or margarine; 3 1/2 c. bite size shredded rice biscuits
<INSTRUCTIONS> 1. In a heavy 2-quart saucepan, mix brown sugar, nuts, evaporated milk and butter or margarine. 2. Stir over medium heat until mixture bubbles all over top. 3. Boil and stir 5 minutes m...


## 10. Create Training/Validation Split

In [16]:
if 'df_clean' in locals():
    # Split into train/validation (80/20)
    train_size = int(0.8 * len(df_clean))
    df_shuffled = df_clean.sample(frac=1, random_state=42).reset_index(drop=True)
    
    df_train = df_shuffled[:train_size]
    df_val = df_shuffled[train_size:]
    
    print(f"📊 Dataset Split:")
    print(f"   - Training: {len(df_train):,} recipes ({len(df_train)/len(df_clean)*100:.1f}%)")
    print(f"   - Validation: {len(df_val):,} recipes ({len(df_val)/len(df_clean)*100:.1f}%)")
    print(f"   - Total: {len(df_clean):,} recipes")
else:
    print("⚠️ No data to split")

📊 Dataset Split:
   - Training: 6,330 recipes (80.0%)
   - Validation: 1,583 recipes (20.0%)
   - Total: 7,913 recipes


## 11. Save Processed Dataset

In [17]:
if 'df_train' in locals() and 'df_val' in locals():
    print("💾 Saving processed datasets...\n")
    
    # Save as JSON
    train_file = DATA_PROCESSED / "train_recipes.json"
    val_file = DATA_PROCESSED / "val_recipes.json"
    full_file = DATA_PROCESSED / "full_recipes.json"
    
    # Save full dataset (structured format)
    full_recipes = df_clean.to_dict('records')
    with open(full_file, 'w', encoding='utf-8') as f:
        json.dump(full_recipes, f, indent=2, ensure_ascii=False)
    
    # Save train/val splits as formatted text
    with open(train_file, 'w', encoding='utf-8') as f:
        json.dump(df_train['formatted_text'].tolist(), f, indent=2, ensure_ascii=False)
    
    with open(val_file, 'w', encoding='utf-8') as f:
        json.dump(df_val['formatted_text'].tolist(), f, indent=2, ensure_ascii=False)
    
    # Save as CSV for easy viewing
    csv_file = DATA_PROCESSED / "recipes_dataset.csv"
    df_clean.to_csv(csv_file, index=False, encoding='utf-8')
    
    print("✅ Dataset saved successfully!")
    print(f"\n📁 Files created:")
    print(f"   - Full dataset: {full_file} ({full_file.stat().st_size / (1024**2):.1f} MB)")
    print(f"   - Training set: {train_file} ({train_file.stat().st_size / (1024**2):.1f} MB)")
    print(f"   - Validation set: {val_file} ({val_file.stat().st_size / (1024**2):.1f} MB)")
    print(f"   - CSV export: {csv_file} ({csv_file.stat().st_size / (1024**2):.1f} MB)")
else:
    print("⚠️ No data to save")

💾 Saving processed datasets...

✅ Dataset saved successfully!

📁 Files created:
   - Full dataset: c:\Users\Champion\Documents\GitHub\cAIuldron\data\processed\recipes\full_recipes.json (13.0 MB)
   - Training set: c:\Users\Champion\Documents\GitHub\cAIuldron\data\processed\recipes\train_recipes.json (3.8 MB)
   - Validation set: c:\Users\Champion\Documents\GitHub\cAIuldron\data\processed\recipes\val_recipes.json (0.9 MB)
   - CSV export: c:\Users\Champion\Documents\GitHub\cAIuldron\data\processed\recipes\recipes_dataset.csv (9.9 MB)


## 12. Generate Dataset Statistics

In [18]:
if 'df_clean' in locals():
    # Generate statistics
    stats = {
        'total_recipes': len(df_clean),
        'unique_ingredients': df_clean['ingredient'].nunique(),
        'unique_cuisines': df_clean['cuisine'].nunique(),
        'ingredient_counts': df_clean['ingredient'].value_counts().head(20).to_dict(),
        'cuisine_counts': df_clean['cuisine'].value_counts().to_dict(),
        'difficulty_counts': df_clean['difficulty'].value_counts().to_dict(),
        'avg_cooking_time': float(df_clean['cooking_time_minutes'].mean()),
        'avg_servings': float(df_clean['servings'].mean()),
        'avg_ingredients_per_recipe': float(df_clean['ingredients'].apply(len).mean()),
        'avg_steps_per_recipe': float(df_clean['instructions'].apply(len).mean()),
        'train_size': len(df_train),
        'val_size': len(df_val),
        'dataset_source': dataset_type
    }
    
    # Save statistics
    stats_file = DATA_PROCESSED / "dataset_statistics.json"
    with open(stats_file, 'w') as f:
        json.dump(stats, f, indent=2)
    
    print("📊 Dataset Statistics Report:")
    print("=" * 80)
    print(json.dumps(stats, indent=2))
    print("=" * 80)
    print(f"\n✅ Statistics saved to: {stats_file}")
else:
    print("⚠️ No data for statistics")

📊 Dataset Statistics Report:
{
  "total_recipes": 7913,
  "unique_ingredients": 205,
  "unique_cuisines": 10,
  "ingredient_counts": {
    "eggs": 1768,
    "c.": 713,
    "chicken": 597,
    "onion": 542,
    "/": 514,
    "ground beef": 379,
    "pepper": 309,
    "chicken breast": 278,
    "(": 268,
    ".": 222,
    "beef": 207,
    "potato": 185,
    "tomato": 162,
    "pork": 160,
    "p.": 125,
    "garlic": 124,
    "broccoli": 92,
    "are": 89,
    "carrot": 83,
    "shrimp": 81
  },
  "cuisine_counts": {
    "american": 6327,
    "italian": 644,
    "asian": 328,
    "mexican": 264,
    "french": 119,
    "indian": 90,
    "chinese": 72,
    "mediterranean": 61,
    "japanese": 7,
    "thai": 1
  },
  "difficulty_counts": {
    "easy": 6354,
    "medium": 1507,
    "hard": 52
  },
  "avg_cooking_time": 30.0,
  "avg_servings": 4.0,
  "avg_ingredients_per_recipe": 7.667382787817515,
  "avg_steps_per_recipe": 6.021862757487678,
  "train_size": 6330,
  "val_size": 1583,
  "datas

## 13. Summary

### ✅ Completed:
1. ✅ Downloaded real recipe dataset (RecipeNLG or Food.com)
2. ✅ Cleaned and standardized data
3. ✅ Extracted main ingredients
4. ✅ Inferred cuisine and difficulty metadata
5. ✅ Formatted recipes for GPT-2 training
6. ✅ Created train/validation splits (80/20)
7. ✅ Saved processed datasets (JSON, CSV)
8. ✅ Generated comprehensive statistics

### 📊 Dataset Sources:
- **RecipeNLG**: 2.2M+ recipes from various sources
- **Food.com**: 500k+ recipes with ratings and reviews

### 💡 Data Format:
```
<INGREDIENT> chicken breast
<TITLE> Grilled Chicken Breast with Herbs
<CUISINE> american
<DIFFICULTY> easy
<TIME> 25 minutes
<SERVINGS> 4
<INGREDIENTS> chicken breasts; olive oil; herbs; salt
<INSTRUCTIONS> 1. Preheat grill... 2. Season chicken...
<END>
```

### 📈 Next Steps:
1. Fine-tune GPT-2 on this dataset
2. Evaluate generation quality
3. Expand to more diverse ingredients
4. Integrate with recipe generation pipeline

In [19]:
if 'df_clean' in locals():
    print("🎉 Recipe Dataset Preparation Complete!")
    print(f"\n📁 Processed data location: {DATA_PROCESSED}")
    print(f"📊 Total recipes: {len(df_clean):,}")
    print(f"📚 Training recipes: {len(df_train):,}")
    print(f"✅ Validation recipes: {len(df_val):,}")
    print(f"🌍 Unique ingredients: {df_clean['ingredient'].nunique()}")
    print(f"🍴 Cuisines: {df_clean['cuisine'].nunique()}")
    print("\n✅ Ready for model training/inference!")
else:
    print("⚠️ Dataset preparation incomplete. Please check error messages above.")

🎉 Recipe Dataset Preparation Complete!

📁 Processed data location: c:\Users\Champion\Documents\GitHub\cAIuldron\data\processed\recipes
📊 Total recipes: 7,913
📚 Training recipes: 6,330
✅ Validation recipes: 1,583
🌍 Unique ingredients: 205
🍴 Cuisines: 10

✅ Ready for model training/inference!
